In [1]:
import requests
import pandas as pd
import json
import datetime as dt

In [2]:
def binance_futures_data(symbol, interval, startTime, endTime, chunk_size=1500):
    url = "https://fapi.binance.com/fapi/v1/klines"  # Binance Futures API URL
    df_list = []
    
    start_time = startTime
    while start_time < endTime:
        end_time = start_time + pd.DateOffset(days=1)
        end_time = min(end_time, endTime)
        
        start_time_ms = int(start_time.timestamp() * 1000)
        end_time_ms = int(end_time.timestamp() * 1000)
        
        req_params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": str(start_time_ms),
            "endTime": str(end_time_ms),
            "limit": str(chunk_size)
        }
        
        response = requests.get(url, params=req_params)
        if response.status_code == 200:
            data = json.loads(response.text)
            if len(data) > 0:
                df = pd.DataFrame(data)
                df = df.iloc[:, 0:6]
                df.columns = ["opentime", "open", "high", "low", "close", "volume"]
                df.index = [dt.datetime.fromtimestamp(x / 1000.0) for x in df.opentime]
                df.drop(columns=['opentime'], inplace=True)
                df.insert(0, "time", df.index)
                df_list.append(df)
        
        start_time = end_time
    
    if not df_list:
        return None
    
    final_df = pd.concat(df_list)
    return final_df

# Example usage
symbol = "BTCUSDT"  # Example trading pair symbol
interval = "1m"    # Example interval (1 hour)
startTime = pd.Timestamp("2019-09-19")  # Example start time
endTime = pd.Timestamp("2020-01-1")    # Example end time

data = binance_futures_data(symbol, interval, startTime, endTime)


In [3]:
data

,time,open,high,low,close,volume
2019-09-19 05:30:00,2019-09-19 05:30:00,10155.16,10155.16,10155.16,10155.16,0
2019-09-19 05:31:00,2019-09-19 05:31:00,10154.70,10155.96,10145,10155.96,29.170
2019-09-19 05:32:00,2019-09-19 05:32:00,10156.71,10159.12,10151.36,10159.12,13.879
2019-09-19 05:33:00,2019-09-19 05:33:00,10159.15,10166.40,10159.15,10165.55,10.694
2019-09-19 05:34:00,2019-09-19 05:34:00,10165.55,10165.55,10159.43,10160.52,10.048
...,...,...,...,...,...,...
2020-01-01 05:26:00,2020-01-01 05:26:00,7199.69,7199.98,7187.96,7189.27,145.662
2020-01-01 05:27:00,2020-01-01 05:27:00,7187.40,7189.49,7185.87,7186.90,24.952
2020-01-01 05:28:00,2020-01-01 05:28:00,7186.89,7189,7186.89,7189,74.694
2020-01-01 05:29:00,2020-01-01 05:29:00,7189,7189.54,7188.88,7189,22.243


In [5]:
data.to_csv("btcf1m19_09_19to1_1_20.csv",index=False)